# Conquer Online 5039 — read-only map inventory
This notebook downloads the historical client into a temporary Colab environment and **lists** installer contents with 7-Zip. It never executes the Windows executable, extracts files, or uploads proprietary assets. Download only the small JSON report. Run at your own discretion in a disposable runtime; installer metadata and original files may still be unsafe or proprietary.
If the archive download is unavailable, stop and provide the error; do not execute the installer.


In [ ]:
# Read-only Conquer 5039 archive inspection. Run in Google Colab, NOT as a game installer.
import os, pathlib, re, subprocess, hashlib, json, shutil, urllib.request
from google.colab import files

URL = "https://archive.org/download/conquer-online-v5039/Conquer_v5039.exe"
DEST = pathlib.Path("/content/Conquer_v5039.exe")
EXPECTED_SIZE = 456150213
print("Downloading historical installer to temporary Colab storage, WITHOUT executing it.")
request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0 TQClassic-ReadOnlyInspector"})
with urllib.request.urlopen(request, timeout=90) as response, DEST.open("wb") as out:
    while True:
        chunk = response.read(1024 * 1024)
        if not chunk:
            break
        out.write(chunk)
print("Size:", DEST.stat().st_size, "bytes")
if DEST.stat().st_size != EXPECTED_SIZE:
    raise ValueError("Unexpected installer size. Stop; source may be incomplete or different.")
with DEST.open("rb") as fh:
    magic=fh.read(2)
assert magic == b"MZ", "Not a Windows executable: stop."
h=hashlib.sha256()
with DEST.open("rb") as fh:
    for chunk in iter(lambda: fh.read(1024*1024), b""):
        h.update(chunk)
print("SHA256:", h.hexdigest())

# Archive lister only: 7-Zip inspects embedded files, it does NOT execute Conquer_v5039.exe.
subprocess.run(["apt-get","update","-qq"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["apt-get","install","-y","-qq","p7zip-full"], check=True, stdout=subprocess.DEVNULL)
listing = subprocess.run(["7z","l","-slt",str(DEST)], capture_output=True, text=True,
                         errors="replace", timeout=240, check=False)
print("Archive lister exit:", listing.returncode)
if listing.returncode:
    print(listing.stderr[-1200:])
    print(listing.stdout[-1200:])
    raise RuntimeError("7-Zip cannot list this installer; report this result and do not run the EXE.")
names=[m.group(1) for m in re.finditer(r"^Path = (.+)$", listing.stdout, re.MULTILINE)]
if names and names[0].endswith("Conquer_v5039.exe"): names=names[1:]
map_candidates=[n for n in names if re.search(
    r"(?:\.dmap$|gamemap\.(?:ini|dat)$|(?:^|[\\/])map(?:[\\/]|$)|(?:^|[\\/])scene(?:[\\/]|$)|minimap)",
    n, re.IGNORECASE)]
print("Listed entries:",len(names),"; map-related entries:",len(map_candidates))
for name in map_candidates[:120]: print(name)
report={"source":URL,"size_bytes":DEST.stat().st_size,"sha256":h.hexdigest(),
        "entries":len(names),"map_candidate_count":len(map_candidates),
        "map_candidates":map_candidates[:500],"status":"metadata_only_no_execution"}
path=pathlib.Path("/content/conquer_5039_map_inventory.json")
path.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Inspection complete. Upload ONLY the small JSON result back to ChatGPT.")
files.download(str(path))
